# Milestone 8 — Training Pipeline

This notebook verifies the Trainer implementation by performing model and dataloader creation, training/validation iterations, and checkpoint save/load operations.

## 1. Environment Setup & Imports

In [ ]:
import sys
from pathlib import Path
import torch

# Add project root to sys.path to allow importing src
project_root = str(Path("..").resolve())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.models.detection.baseline_detector import BaselineDetector
from src.datasets.dataloader import BiohubDataLoader
from src.losses.detection_loss import DetectionLoss
from src.trainer.trainer import Trainer
print("Imports loaded successfully.")

## 2. Model Creation

In [ ]:
model = BaselineDetector(in_channels=1, hidden_channels=16, out_channels=1)
print("Model created:")
print(model)

## 3. DataLoader Creation

In [ ]:
# Find first matching dataset pair programmatically
train_dir = Path("../data/raw/train")
zarr_paths = sorted(list(train_dir.glob("*.zarr")))
geff_paths = sorted(list(train_dir.glob("*.geff")))

test_zarr_path = None
test_geff_path = None

for z_path in zarr_paths:
    g_path = z_path.with_suffix(".geff")
    if g_path in geff_paths:
        test_zarr_path = z_path
        test_geff_path = g_path
        break

if not test_zarr_path or not test_geff_path:
    raise FileNotFoundError("Matching Zarr/GEFF pair not found in data/raw/train")

print(f"Zarr path: {test_zarr_path}")
print(f"GEFF path: {test_geff_path}")

bh_loader = BiohubDataLoader(
    test_zarr_path,
    test_geff_path,
    batch_size=1,
    shuffle=False,
)
train_loader = bh_loader.get_train_loader()
val_loader = bh_loader.get_train_loader()
print("Train and Validation DataLoaders created.")

## 4. Trainer Creation

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    stream=sys.stdout
)

loss_fn = DetectionLoss()
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    lr=1e-3,
    step_size=10,
    gamma=0.1,
    device="cpu"
)
print("Trainer created successfully.")

## 5. One Training Iteration

In [ ]:
batch = next(iter(train_loader))
train_loss = trainer.train_step(batch)
print(f"Training step loss: {train_loss:.6f}")

## 6. One Validation Iteration

In [ ]:
val_loss = trainer.val_step(batch)
print(f"Validation step loss: {val_loss:.6f}")

## 7. Checkpoint Save

In [ ]:
checkpoint_path = Path("test_checkpoint.pt")
trainer.epoch = 1
trainer.save_checkpoint(checkpoint_path)
print(f"Checkpoint file exists: {checkpoint_path.exists()}")

## 8. Checkpoint Load

In [ ]:
# Reset trainer state
trainer.epoch = 0

# Load checkpoint
trainer.load_checkpoint(checkpoint_path)
print(f"Restored epoch from checkpoint: {trainer.epoch}")

# Clean up the temporary checkpoint file
if checkpoint_path.exists():
    checkpoint_path.unlink()
    print("Temporary checkpoint file cleaned up.")